## Carico il BRT

In [1]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root.resolve())

Project root: /home/alessio/TESI/hj_reachability


In [2]:
# Percorso del BRT salvato localmente
data_path = project_root / "results" / "brt_euclidean.npz"

if not data_path.exists():
    raise FileNotFoundError(
        f"File non trovato:\n{data_path.resolve()}"
    )

data = np.load(data_path)

print("File caricato:", data_path.resolve())
print("Variabili disponibili:")
for name in data.files:
    print(f"  - {name}")

File caricato: /home/alessio/TESI/hj_reachability/results/brt_euclidean.npz
Variabili disponibili:
  - BRT
  - gradienti
  - x_rel
  - y_rel
  - theta_rel
  - v_H
  - delta_E
  - v_E
  - target_time
  - periodic_dims


## Estrarre i dati

In [3]:
BRT = data["BRT"]
gradienti = data["gradienti"]

x_rel_grid = data["x_rel"]
y_rel_grid = data["y_rel"]
theta_rel_grid = data["theta_rel"]
v_H_grid = data["v_H"]
delta_E_grid = data["delta_E"]
v_E_grid = data["v_E"]

target_time = float(data["target_time"])
periodic_dims = data["periodic_dims"]

coordinate_vectors = (
    x_rel_grid,
    y_rel_grid,
    theta_rel_grid,
    v_H_grid,
    delta_E_grid,
    v_E_grid,
)

In [4]:
grid_shape = tuple(len(vector) for vector in coordinate_vectors)

print("Forma ricostruita della griglia:", grid_shape)
print("Forma BRT:", BRT.shape)
print("Forma gradienti:", gradienti.shape)
print("Orizzonte BRT:", target_time, "s")
print("Dimensioni periodiche:", periodic_dims)

assert BRT.shape == grid_shape
assert gradienti.shape == (*grid_shape, 6)

print("\nCaricamento completato correttamente.")

Forma ricostruita della griglia: (26, 13, 15, 11, 11, 11)
Forma BRT: (26, 13, 15, 11, 11, 11)
Forma gradienti: (26, 13, 15, 11, 11, 11, 6)
Orizzonte BRT: -3.0 s
Dimensioni periodiche: [2]

Caricamento completato correttamente.


## Controllo numerico

In [5]:
n_nan_BRT = np.isnan(BRT).sum()
n_inf_BRT = np.isinf(BRT).sum()

n_nan_gradienti = np.isnan(gradienti).sum()
n_inf_gradienti = np.isinf(gradienti).sum()

print("BRT")
print("  NaN:", n_nan_BRT)
print("  Inf:", n_inf_BRT)

print("\nGradienti")
print("  NaN:", n_nan_gradienti)
print("  Inf:", n_inf_gradienti)

assert n_nan_BRT == 0
assert n_inf_BRT == 0
assert n_nan_gradienti == 0
assert n_inf_gradienti == 0

print("\nBRT e gradienti contengono soltanto valori finiti.")

BRT
  NaN: 0
  Inf: 0

Gradienti
  NaN: 0
  Inf: 0

BRT e gradienti contengono soltanto valori finiti.


In [7]:
# Controlli sul BRT

print(f"Minimo BRT: {BRT.min():.6f}")
print(f"Massimo BRT: {BRT.max():.6f}")

n_inside = np.count_nonzero(BRT <= 0)
n_outside = np.count_nonzero(BRT > 0)
n_total = BRT.size

print("\nDistribuzione del segno")
print(f"  BRT <= 0: {n_inside:,} punti ({100 * n_inside / n_total:.2f}%)")
print(f"  BRT > 0:  {n_outside:,} punti ({100 * n_outside / n_total:.2f}%)")

Minimo BRT: -23.305016
Massimo BRT: 13.482062

Distribuzione del segno
  BRT <= 0: 2,305,629 punti (34.17%)
  BRT > 0:  4,442,541 punti (65.83%)


In [8]:
# Controlli sul gradiente

gradient_names = (
    "dV/dx_rel",
    "dV/dy_rel",
    "dV/dtheta_rel",
    "dV/dv_H",
    "dV/ddelta_E",
    "dV/dv_E",
)

for dimension, name in enumerate(gradient_names):
    component = gradienti[..., dimension]

    print(f"{name}")
    print(f"  minimo:  {component.min(): .6f}")
    print(f"  massimo: {component.max(): .6f}")
    print()

dV/dx_rel
  minimo:  -9.917418
  massimo:  12.389023

dV/dy_rel
  minimo:  -12.057384
  massimo:  10.449586

dV/dtheta_rel
  minimo:  -19.598701
  massimo:  21.568453

dV/dv_H
  minimo:  -2.603590
  massimo:  6.920338

dV/ddelta_E
  minimo:  -78.786095
  massimo:  77.436363

dV/dv_E
  minimo:  -2.204033
  massimo:  12.624368



In [9]:
# Controlli sulla griglia

coordinate_names = (
    "x_rel",
    "y_rel",
    "theta_rel",
    "v_H",
    "delta_E",
    "v_E",
)

for name, vector in zip(coordinate_names, coordinate_vectors):
    steps = np.diff(vector)

    print(name)
    print(f"  punti:   {len(vector)}")
    print(f"  minimo:  {vector.min():.6f}")
    print(f"  massimo: {vector.max():.6f}")
    print(f"  passo minimo: {steps.min():.6f}")
    print(f"  passo massimo: {steps.max():.6f}")
    print()

x_rel
  punti:   26
  minimo:  -8.000000
  massimo: 17.000000
  passo minimo: 1.000000
  passo massimo: 1.000000

y_rel
  punti:   13
  minimo:  -6.000000
  massimo: 6.000000
  passo minimo: 1.000000
  passo massimo: 1.000000

theta_rel
  punti:   15
  minimo:  -0.785398
  massimo: 0.680679
  passo minimo: 0.104720
  passo massimo: 0.104720

v_H
  punti:   11
  minimo:  1.000000
  massimo: 11.000000
  passo minimo: 1.000000
  passo massimo: 1.000000

delta_E
  punti:   11
  minimo:  -0.261799
  massimo: 0.261799
  passo minimo: 0.052360
  passo massimo: 0.052360

v_E
  punti:   11
  minimo:  1.000000
  massimo: 11.000000
  passo minimo: 1.000000
  passo massimo: 1.000000



## Interpolatore BRT

In [10]:
from scipy.interpolate import RegularGridInterpolator

In [11]:
# Interpolatore del BRT

BRT_interpolator = RegularGridInterpolator(
    points=coordinate_vectors,
    values=BRT,
    method="linear",
    bounds_error=True,
)

print("Interpolatore BRT costruito correttamente.")

Interpolatore BRT costruito correttamente.


In [14]:
def evaluate_BRT(state):
    """
    Valuta il BRT nello stato relativo 6D.

    Ordine:
    [x_rel, y_rel, theta_rel, v_H, delta_E, v_E]
    """
    state = np.asarray(state, dtype=float)

    if state.shape != (6,):
        raise ValueError(
            f"Lo stato deve avere forma (6,), ricevuta {state.shape}."
        )

    interpolated_value = BRT_interpolator(state)

    return np.asarray(interpolated_value).item()

In [15]:
# Indici di un nodo circa centrale nella griglia

central_indices = tuple(
    len(vector) // 2
    for vector in coordinate_vectors
)

central_state = np.array(
    [
        vector[index]
        for vector, index in zip(
            coordinate_vectors,
            central_indices,
        )
    ]
)

interpolated_value = evaluate_BRT(central_state)
stored_value = float(BRT[central_indices])

print("Indici:", central_indices)
print("Stato centrale:", central_state)
print("Valore salvato:     ", stored_value)
print("Valore interpolato: ", interpolated_value)
print("Errore assoluto:    ", abs(interpolated_value - stored_value))

Indici: (13, 6, 7, 5, 5, 5)
Stato centrale: [ 5.          0.         -0.05235982  6.          0.          6.        ]
Valore salvato:      -1.0378766059875488
Valore interpolato:  -1.0378766059875488
Errore assoluto:     0.0


## Interpolatore gradiente

In [16]:
# Interpolatore del gradiente del BRT

gradient_interpolator = RegularGridInterpolator(
    points=coordinate_vectors,
    values=gradienti,
    method="linear",
    bounds_error=True,
)

print("Interpolatore del gradiente costruito correttamente.")

Interpolatore del gradiente costruito correttamente.


In [17]:
def evaluate_gradient(state):
    """
    Valuta il gradiente 6D del BRT nello stato fornito.

    Ordine dello stato:
    [x_rel, y_rel, theta_rel, v_H, delta_E, v_E]
    """
    state = np.asarray(state, dtype=float)

    if state.shape != (6,):
        raise ValueError(
            f"Lo stato deve avere forma (6,), ricevuta {state.shape}."
        )

    gradient = np.asarray(
        gradient_interpolator(state),
        dtype=float,
    )

    # L'interpolatore può restituire forma (1, 6)
    return gradient.reshape(6)

In [18]:
interpolated_gradient = evaluate_gradient(central_state)
stored_gradient = np.asarray(
    gradienti[central_indices],
    dtype=float,
)

print("Stato centrale:")
print(central_state)

print("\nGradiente salvato:")
print(stored_gradient)

print("\nGradiente interpolato:")
print(interpolated_gradient)

print("\nErrore assoluto per componente:")
print(np.abs(interpolated_gradient - stored_gradient))

Stato centrale:
[ 5.          0.         -0.05235982  6.          0.          6.        ]

Gradiente salvato:
[ 1.38487029 -0.06282669 -0.15580286  0.7445389   1.15731454 -0.6764046 ]

Gradiente interpolato:
[ 1.38487029 -0.06282669 -0.15580286  0.7445389   1.15731454 -0.6764046 ]

Errore assoluto per componente:
[0. 0. 0. 0. 0. 0.]


In [19]:
for name, value in zip(
    gradient_names,
    interpolated_gradient,
):
    print(f"{name:16s} = {value: .6f}")

dV/dx_rel        =  1.384870
dV/dy_rel        = -0.062827
dV/dtheta_rel    = -0.155803
dV/dv_H          =  0.744539
dV/ddelta_E      =  1.157315
dV/dv_E          = -0.676405


## Dinamica

In [20]:
import jax.numpy as jnp

from hj_reachability.systems.relative_vehicle_6d import RelativeVehicle6D

In [21]:
dynamics = RelativeVehicle6D(
    lf=1.2,
    lr=1.5,
)

print("Dinamica RelativeVehicle6D costruita correttamente.")

Dinamica RelativeVehicle6D costruita correttamente.


In [22]:
print("Limiti controllo Ego:")
print("  minimo:", np.asarray(dynamics.control_space.lo))
print("  massimo:", np.asarray(dynamics.control_space.hi))

print("\nLimiti disturbo Human:")
print("  minimo:", np.asarray(dynamics.disturbance_space.lo))
print("  massimo:", np.asarray(dynamics.disturbance_space.hi))

Limiti controllo Ego:
  minimo: [-0.087 -7.   ]
  massimo: [0.087 2.5  ]

Limiti disturbo Human:
  minimo: [-1. -7.]
  massimo: [1.  2.5]


In [23]:
test_state = jnp.array([
    5.0,                 # x_rel [m]
    1.0,                 # y_rel [m]
    np.deg2rad(10.0),    # theta_rel [rad]
    12.0,                # v_H [m/s]
    0.05,                # delta_E [rad]
    14.0,                # v_E [m/s]
])

test_time = 0.0

In [24]:
open_loop = dynamics.open_loop_dynamics(
    test_state,
    test_time,
)

control_jacobian = dynamics.control_jacobian(
    test_state,
    test_time,
)

disturbance_jacobian = dynamics.disturbance_jacobian(
    test_state,
    test_time,
)

print("Dinamica libera f(x):")
print(np.asarray(open_loop))

print("\nMatrice di controllo G(x):")
print(np.asarray(control_jacobian))

print("\nMatrice di disturbo H(x):")
print(np.asarray(disturbance_jacobian))

Dinamica libera f(x):
[-1.9175246  0.3978386 -0.2593753  0.         0.         0.       ]

Matrice di controllo G(x):
[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 1.]]

Matrice di disturbo H(x):
[[0. 0.]
 [0. 0.]
 [1. 0.]
 [0. 1.]
 [0. 0.]
 [0. 0.]]


In [25]:
assert open_loop.shape == (6,)
assert control_jacobian.shape == (6, 2)
assert disturbance_jacobian.shape == (6, 2)

print("\nLe dimensioni della dinamica sono corrette.")


Le dimensioni della dinamica sono corrette.


## Ricerca di uno stato iniziale libero

In [26]:
import hj_reachability as hj

from hj_reachability.vehicle.geometry import build_terminal_set_boundary
from hj_reachability.vehicle.metrics import metricEuclidean

In [27]:
# costruzione griglia
grid = hj.Grid.from_lattice_parameters_and_boundary_conditions(
    domain=hj.sets.Box(
        lo=jnp.array([
            -8.0,
            -6.0,
            -0.25 * np.pi,
            1.0,
            -np.pi / 12,
            1.0,
        ]),
        hi=jnp.array([
            17.0,
            6.0,
            0.25 * np.pi,
            11.0,
            np.pi / 12,
            11.0,
        ]),
    ),
    shape=grid_shape,
    periodic_dims=2,
)

In [28]:
# verifico che coincida con la griglia caricata
for name, reconstructed, loaded in zip(
    coordinate_names,
    grid.coordinate_vectors,
    coordinate_vectors,
):
    reconstructed = np.asarray(reconstructed)

    assert np.allclose(
        reconstructed,
        loaded,
        rtol=1e-6,
        atol=1e-6,
    ), f"Coordinate non coincidenti per {name}"

print("La griglia ricostruita coincide con quella salvata.")

La griglia ricostruita coincide con quella salvata.


In [29]:
# ricalcolo la condizione terminale
boundary = build_terminal_set_boundary(
    n_theta=90,
    n_phi=180,
)

metric_result = metricEuclidean(
    grid=grid,
    boundary=boundary,
)

V0 = np.asarray(metric_result.terminal_values)

assert V0.shape == BRT.shape

print("Forma V0:", V0.shape)
print(f"Minimo V0: {V0.min():.6f}")
print(f"Massimo V0: {V0.max():.6f}")

Forma V0: (26, 13, 15, 11, 11, 11)
Minimo V0: -3.204360
Massimo V0: 13.482062


In [67]:
# cerco dei punti fuori dal terminal set
brt_tolerance = 1e-6
minimum_clearance = 0.25

interior_mask = np.ones(BRT.shape, dtype=bool)

# Escludiamo due nodi da ciascun bordo non periodico
boundary_nodes = 2

for dimension, vector in enumerate(coordinate_vectors):
    # theta_rel è periodica
    if dimension == 2:
        continue

    valid_coordinates = np.zeros(len(vector), dtype=bool)
    valid_coordinates[
        boundary_nodes:-boundary_nodes
    ] = True

    reshape_shape = [1] * BRT.ndim
    reshape_shape[dimension] = len(vector)

    interior_mask &= valid_coordinates.reshape(reshape_shape)

candidate_mask = (
    (BRT < -brt_tolerance)
    & (V0 > minimum_clearance)
    & interior_mask
)

n_candidates = np.count_nonzero(candidate_mask)
print("Candidati interni:", f"{n_candidates:,}")

if n_candidates == 0:
    raise RuntimeError(
        "Nessun candidato soddisfa i vincoli scelti."
    )

Candidati interni: 222,152


In [68]:
candidate_values = np.where(
    candidate_mask,
    BRT,
    np.inf,
)

initial_indices = np.unravel_index(
    np.argmin(candidate_values),
    BRT.shape,
)

initial_state = np.array(
    [
        vector[index]
        for vector, index in zip(
            coordinate_vectors,
            initial_indices,
        )
    ],
    dtype=float,
)

In [70]:
initial_BRT = evaluate_BRT(initial_state)
initial_gradient = evaluate_gradient(initial_state)
initial_V0 = float(V0[initial_indices])

print("Indici iniziali:", initial_indices)

print("\nStato iniziale:")
for name, value in zip(coordinate_names, initial_state):
    print(f"  {name:10s} = {value: .6f}")

print(f"\nBRT(x0) = {initial_BRT:.6f}")
print(f"V0(x0)  = {initial_V0:.6f}")

print("\nGradiente iniziale:")
for name, value in zip(gradient_names, initial_gradient):
    print(f"  {name:16s} = {value: .6f}")

Indici iniziali: (np.int64(12), np.int64(8), np.int64(3), np.int64(2), np.int64(8), np.int64(2))

Stato iniziale:
  x_rel      =  4.000000
  y_rel      =  2.000000
  theta_rel  = -0.471239
  v_H        =  3.000000
  delta_E    =  0.157080
  v_E        =  3.000000

BRT(x0) = -3.926488
V0(x0)  = 0.651295

Gradiente iniziale:
  dV/dx_rel        =  1.230018
  dV/dy_rel        =  0.381645
  dV/dtheta_rel    =  0.049725
  dV/dv_H          =  0.392815
  dV/ddelta_E      = -3.628397
  dV/dv_E          =  1.326654


## Calcolo del controllo ottimo

In [71]:
# Conversione in array JAX

state_jax = jnp.asarray(initial_state)
gradient_jax = jnp.asarray(initial_gradient)

initial_time = 0.0


optimal_control, optimal_disturbance = (
    dynamics.optimal_control_and_disturbance(
        state_jax,
        initial_time,
        gradient_jax,
    )
)

optimal_control = np.asarray(optimal_control, dtype=float)
optimal_disturbance = np.asarray(optimal_disturbance, dtype=float)

In [72]:
print("Controllo ottimo Ego:")
print(f"  steering_rate_E = {optimal_control[0]: .6f} rad/s")
print(f"  acceleration_E  = {optimal_control[1]: .6f} m/s²")

print("\nDisturbo ottimo Human:")
print(f"  yaw_rate_H      = {optimal_disturbance[0]: .6f} rad/s")
print(f"  acceleration_H  = {optimal_disturbance[1]: .6f} m/s²")

Controllo ottimo Ego:
  steering_rate_E = -0.087000 rad/s
  acceleration_E  =  2.500000 m/s²

Disturbo ottimo Human:
  yaw_rate_H      = -1.000000 rad/s
  acceleration_H  = -7.000000 m/s²


In [73]:
control_lower = np.asarray(dynamics.control_space.lo)
control_upper = np.asarray(dynamics.control_space.hi)

disturbance_lower = np.asarray(dynamics.disturbance_space.lo)
disturbance_upper = np.asarray(dynamics.disturbance_space.hi)

assert np.all(optimal_control >= control_lower)
assert np.all(optimal_control <= control_upper)

assert np.all(optimal_disturbance >= disturbance_lower)
assert np.all(optimal_disturbance <= disturbance_upper)

print("Controllo e disturbo rispettano i rispettivi limiti.")

Controllo e disturbo rispettano i rispettivi limiti.


In [74]:
# calcolo lo stato
open_loop = dynamics.open_loop_dynamics(
    state_jax,
    initial_time,
)

control_jacobian = dynamics.control_jacobian(
    state_jax,
    initial_time,
)

disturbance_jacobian = dynamics.disturbance_jacobian(
    state_jax,
    initial_time,
)

state_derivative = (
    open_loop
    + control_jacobian @ jnp.asarray(optimal_control)
    + disturbance_jacobian @ jnp.asarray(optimal_disturbance)
)

state_derivative = np.asarray(state_derivative, dtype=float)

In [75]:
state_derivative_names = (
    "dx_rel/dt",
    "dy_rel/dt",
    "dtheta_rel/dt",
    "dv_H/dt",
    "ddelta_E/dt",
    "dv_E/dt",
)

print("Derivata dello stato con le azioni ottime:\n")

for name, value in zip(
    state_derivative_names,
    state_derivative,
):
    print(f"  {name:16s} = {value: .6f}")

Derivata dello stato con le azioni ottime:

  dx_rel/dt        =  0.035177
  dy_rel/dt        = -2.326151
  dtheta_rel/dt    = -1.175305
  dv_H/dt          = -7.000000
  ddelta_E/dt      = -0.087000
  dv_E/dt          =  2.500000


In [76]:
# hamiltoniano nel punto iniziale
hamiltonian = float(
    np.dot(initial_gradient, state_derivative)
)

print(f"Hamiltoniana ottima: {hamiltonian:.6f}")

Hamiltoniana ottima: -0.020337


## Prima soluzione manuale del problema min-max

In [77]:
from itertools import product


control_vertices = np.array(
    list(product(
        [control_lower[0], control_upper[0]],
        [control_lower[1], control_upper[1]],
    )),
    dtype=float,
)

disturbance_vertices = np.array(
    list(product(
        [disturbance_lower[0], disturbance_upper[0]],
        [disturbance_lower[1], disturbance_upper[1]],
    )),
    dtype=float,
)

print("Vertici controllo Ego:")
print(control_vertices)

print("\nVertici disturbo Human:")
print(disturbance_vertices)

assert control_vertices.shape == (4, 2)
assert disturbance_vertices.shape == (4, 2)

Vertici controllo Ego:
[[-0.087 -7.   ]
 [-0.087  2.5  ]
 [ 0.087 -7.   ]
 [ 0.087  2.5  ]]

Vertici disturbo Human:
[[-1.  -7. ]
 [-1.   2.5]
 [ 1.  -7. ]
 [ 1.   2.5]]


In [78]:
# calcolo l'hamiltoniano per tutti i vertici
open_loop_np = np.asarray(open_loop, dtype=float)
control_jacobian_np = np.asarray(control_jacobian, dtype=float)
disturbance_jacobian_np = np.asarray(
    disturbance_jacobian,
    dtype=float,
)

hamiltonian_table = np.empty((4, 4))

for control_index, control in enumerate(control_vertices):
    for disturbance_index, disturbance in enumerate(
        disturbance_vertices
    ):
        derivative = (
            open_loop_np
            + control_jacobian_np @ control
            + disturbance_jacobian_np @ disturbance
        )

        hamiltonian_table[
            control_index,
            disturbance_index,
        ] = np.dot(initial_gradient, derivative)

In [79]:
print("Matrice delle Hamiltoniane")
print("Righe: controllo Ego")
print("Colonne: disturbo Human\n")

print(np.round(hamiltonian_table, 6))

Matrice delle Hamiltoniane
Righe: controllo Ego
Colonne: disturbo Human

[[-12.623548  -8.891805 -12.524098  -8.792355]
 [ -0.020337   3.711405   0.079113   3.810855]
 [-13.254889  -9.523146 -13.155439  -9.423696]
 [ -0.651678   3.080064  -0.552228   3.179514]]


In [80]:
worst_case_for_each_control = np.min(
    hamiltonian_table,
    axis=1,
)

worst_disturbance_indices = np.argmin(
    hamiltonian_table,
    axis=1,
)

print("Minimo imposto da Human per ogni controllo Ego:\n")

for index, control in enumerate(control_vertices):
    print(
        f"u = {control}  ->  "
        f"min_d H = {worst_case_for_each_control[index]: .6f}"
    )

Minimo imposto da Human per ogni controllo Ego:

u = [-0.087 -7.   ]  ->  min_d H = -12.623548
u = [-0.087  2.5  ]  ->  min_d H = -0.020337
u = [ 0.087 -7.   ]  ->  min_d H = -13.254889
u = [0.087 2.5  ]  ->  min_d H = -0.651678


In [81]:
manual_control_index = np.argmax(
    worst_case_for_each_control
)

manual_disturbance_index = worst_disturbance_indices[
    manual_control_index
]

manual_optimal_control = control_vertices[
    manual_control_index
]

manual_optimal_disturbance = disturbance_vertices[
    manual_disturbance_index
]

manual_hamiltonian = hamiltonian_table[
    manual_control_index,
    manual_disturbance_index,
]

In [82]:
print("Soluzione max-min calcolata manualmente:\n")

print("Controllo Ego:")
print(manual_optimal_control)

print("\nDisturbo Human:")
print(manual_optimal_disturbance)

print(f"\nHamiltoniana: {manual_hamiltonian:.6f}")

Soluzione max-min calcolata manualmente:

Controllo Ego:
[-0.087  2.5  ]

Disturbo Human:
[-1. -7.]

Hamiltoniana: -0.020337


In [83]:
print("Risultato della classe:")
print("  controllo:", optimal_control)
print("  disturbo: ", optimal_disturbance)
print("  H:        ", hamiltonian)

print("\nRisultato manuale:")
print("  controllo:", manual_optimal_control)
print("  disturbo: ", manual_optimal_disturbance)
print("  H:        ", manual_hamiltonian)

Risultato della classe:
  controllo: [-0.087  2.5  ]
  disturbo:  [-1. -7.]
  H:         -0.020337280882923636

Risultato manuale:
  controllo: [-0.087  2.5  ]
  disturbo:  [-1. -7.]
  H:         -0.020337280882923636


In [84]:
assert np.allclose(
    optimal_control,
    manual_optimal_control,
    atol=1e-7,
)

assert np.allclose(
    optimal_disturbance,
    manual_optimal_disturbance,
    atol=1e-7,
)

assert np.isclose(
    hamiltonian,
    manual_hamiltonian,
    atol=1e-6,
)

print(
    "\nVerifica superata: la classe restituisce "
    "la soluzione max-min dell'Hamiltoniana."
)


Verifica superata: la classe restituisce la soluzione max-min dell'Hamiltoniana.


In [85]:
simulation_time = abs(target_time)
dt_simulation = 0.01

maximum_steps = int(np.ceil(simulation_time / dt_simulation))

state = initial_state.copy()
time = 0.0

grid_lower = np.array(
    [vector[0] for vector in coordinate_vectors],
    dtype=float,
)

grid_upper = np.array(
    [vector[-1] for vector in coordinate_vectors],
    dtype=float,
)

print("Durata simulazione:", simulation_time, "s")
print("Passo temporale:", dt_simulation, "s")
print("Numero massimo di passi:", maximum_steps)

Durata simulazione: 3.0 s
Passo temporale: 0.01 s
Numero massimo di passi: 300


In [86]:
def state_is_inside_grid(state):
    """Controlla se tutte le componenti appartengono alla griglia."""
    state = np.asarray(state, dtype=float)

    return bool(
        np.all(state >= grid_lower)
        and np.all(state <= grid_upper)
    )

def closed_loop_dynamics(state, time):
    """
    Calcola la derivata dello stato usando il gradiente
    del BRT salvato all'orizzonte target_time.
    """
    state = np.asarray(state, dtype=float)

    if not state_is_inside_grid(state):
        raise ValueError("Lo stato è uscito dal dominio della griglia.")

    gradient = evaluate_gradient(state)

    state_jax = jnp.asarray(state)
    gradient_jax = jnp.asarray(gradient)

    control, disturbance = (
        dynamics.optimal_control_and_disturbance(
            state_jax,
            time,
            gradient_jax,
        )
    )

    control = np.asarray(control, dtype=float)
    disturbance = np.asarray(disturbance, dtype=float)

    open_loop = dynamics.open_loop_dynamics(
        state_jax,
        time,
    )

    control_jacobian = dynamics.control_jacobian(
        state_jax,
        time,
    )

    disturbance_jacobian = dynamics.disturbance_jacobian(
        state_jax,
        time,
    )

    derivative = (
        open_loop
        + control_jacobian @ jnp.asarray(control)
        + disturbance_jacobian @ jnp.asarray(disturbance)
    )

    return (
        np.asarray(derivative, dtype=float),
        control,
        disturbance,
        gradient,
    )

In [87]:
time_history = []
state_history = []
BRT_history = []
V0_history = []
control_history = []
disturbance_history = []
gradient_history = []

termination_reason = "maximum simulation time reached"

In [88]:
V0_interpolator = RegularGridInterpolator(
    points=coordinate_vectors,
    values=V0,
    method="linear",
    bounds_error=True,
)


def evaluate_V0(state):
    """Valuta la distanza con segno dal terminal set."""
    state = np.asarray(state, dtype=float)

    if state.shape != (6,):
        raise ValueError(
            f"Lo stato deve avere forma (6,), ricevuta {state.shape}."
        )

    return np.asarray(V0_interpolator(state)).item()

## Simulazione

In [89]:
for step in range(maximum_steps + 1):

    if not state_is_inside_grid(state):
        termination_reason = "state left the interpolation grid"
        break

    current_BRT = evaluate_BRT(state)
    current_V0 = evaluate_V0(state)

    derivative, control, disturbance, gradient = (
        closed_loop_dynamics(state, time)
    )

    time_history.append(time)
    state_history.append(state.copy())
    BRT_history.append(current_BRT)
    V0_history.append(current_V0)
    control_history.append(control.copy())
    disturbance_history.append(disturbance.copy())
    gradient_history.append(gradient.copy())

    # Collisione o contatto
    if current_V0 <= 0.0:
        termination_reason = "terminal set reached"
        break

    # Ultimo istante dell'orizzonte
    if time >= simulation_time:
        break

    # Integrazione di Euler
    state = state + dt_simulation * derivative
    time = min(
        time + dt_simulation,
        simulation_time,
    )

In [90]:
time_history = np.asarray(time_history)
state_history = np.asarray(state_history)
BRT_history = np.asarray(BRT_history)
V0_history = np.asarray(V0_history)
control_history = np.asarray(control_history)
disturbance_history = np.asarray(disturbance_history)
gradient_history = np.asarray(gradient_history)

In [91]:
print("Simulazione terminata.")
print("Motivo:", termination_reason)
print(f"Tempo finale: {time_history[-1]:.3f} s")
print(f"Numero di campioni: {len(time_history)}")

print(f"\nBRT iniziale: {BRT_history[0]:.6f}")
print(f"BRT finale:   {BRT_history[-1]:.6f}")

print(f"\nV0 iniziale: {V0_history[0]:.6f}")
print(f"V0 finale:   {V0_history[-1]:.6f}")

print("\nStato finale:")
for name, value in zip(
    coordinate_names,
    state_history[-1],
):
    print(f"  {name:10s} = {value: .6f}")

Simulazione terminata.
Motivo: terminal set reached
Tempo finale: 0.240 s
Numero di campioni: 25

BRT iniziale: -3.926488
BRT finale:   -3.425308

V0 iniziale: 0.651295
V0 finale:   -0.025959

Stato finale:
  x_rel      =  3.768604
  y_rel      =  1.548907
  theta_rel  = -0.314291
  v_H        =  1.320000
  delta_E    =  0.136200
  v_E        =  3.600000


## Simulazione a partire da BRT positivo

In [106]:
# ============================================================
# TEST COMPLETO: PUNTO CASUALE ESTERNO AL BRT E SIMULAZIONE
# ============================================================

# Impostazioni
random_seed = None          # Inserire un intero per rendere il test ripetibile
minimum_brt_value = 0.25    # Margine minimo esterno al BRT
minimum_clearance = 0.25    # Distanza minima iniziale dalla collisione [m]
boundary_nodes = 2          # Nodi esclusi dai bordi della griglia
simulation_time = abs(target_time)
dt_simulation = 0.01

rng = np.random.default_rng(random_seed)

# ------------------------------------------------------------
# 1. Costruzione della maschera dei punti interni alla griglia
# ------------------------------------------------------------

interior_mask = np.ones(BRT.shape, dtype=bool)

for dimension, vector in enumerate(coordinate_vectors):

    # theta_rel è una dimensione periodica
    if dimension == 2:
        continue

    if len(vector) <= 2 * boundary_nodes:
        raise ValueError(
            f"La dimensione {coordinate_names[dimension]} "
            "ha troppo pochi nodi per escludere i bordi."
        )

    valid_coordinates = np.zeros(len(vector), dtype=bool)
    valid_coordinates[boundary_nodes:-boundary_nodes] = True

    reshape_shape = [1] * BRT.ndim
    reshape_shape[dimension] = len(vector)

    interior_mask &= valid_coordinates.reshape(reshape_shape)

# Punto esterno al BRT, inizialmente fuori dalla collisione
positive_candidate_mask = (
    np.isfinite(BRT)
    & np.isfinite(V0)
    & (BRT > minimum_brt_value)
    & (V0 > minimum_clearance)
    & interior_mask
)

positive_candidate_indices = np.argwhere(positive_candidate_mask)
n_candidates = len(positive_candidate_indices)

if n_candidates == 0:
    raise RuntimeError(
        "Non esistono punti che soddisfano i criteri richiesti. "
        "Prova a ridurre minimum_brt_value o minimum_clearance."
    )

# ------------------------------------------------------------
# 2. Estrazione casuale dello stato iniziale
# ------------------------------------------------------------

random_candidate = rng.integers(n_candidates)
initial_indices = tuple(
    positive_candidate_indices[random_candidate]
)

initial_state = np.array(
    [
        vector[index]
        for vector, index in zip(
            coordinate_vectors,
            initial_indices,
        )
    ],
    dtype=float,
)

initial_BRT = evaluate_BRT(initial_state)
initial_V0 = evaluate_V0(initial_state)

print("=" * 60)
print("STATO INIZIALE CASUALE ESTERNO AL BRT")
print("=" * 60)

print(f"Candidati disponibili: {n_candidates:,}")
print("Indici selezionati:", initial_indices)

for name, value in zip(coordinate_names, initial_state):
    print(f"  {name:10s} = {value: .6f}")

print(f"\nBRT(x0) = {initial_BRT:.6f}")
print(f"V0(x0)  = {initial_V0:.6f}")

assert initial_BRT > 0.0
assert initial_V0 > 0.0
assert state_is_inside_grid(initial_state)

# ------------------------------------------------------------
# 3. Inizializzazione della simulazione
# ------------------------------------------------------------

maximum_steps = int(
    np.ceil(simulation_time / dt_simulation)
)

state = initial_state.copy()
time = 0.0

time_history = []
state_history = []
BRT_history = []
V0_history = []
control_history = []
disturbance_history = []
gradient_history = []

termination_reason = "maximum simulation time reached"

# ------------------------------------------------------------
# 4. Simulazione closed-loop
# ------------------------------------------------------------

for step in range(maximum_steps + 1):

    if not state_is_inside_grid(state):
        termination_reason = "state left the interpolation grid"
        break

    current_BRT = evaluate_BRT(state)
    current_V0 = evaluate_V0(state)

    derivative, control, disturbance, gradient = (
        closed_loop_dynamics(state, time)
    )

    time_history.append(time)
    state_history.append(state.copy())
    BRT_history.append(current_BRT)
    V0_history.append(current_V0)
    control_history.append(control.copy())
    disturbance_history.append(disturbance.copy())
    gradient_history.append(gradient.copy())

    if current_V0 <= 0.0:
        termination_reason = "terminal set reached"
        break

    if time >= simulation_time:
        termination_reason = "maximum simulation time reached"
        break

    state = state + dt_simulation * derivative
    time = min(
        time + dt_simulation,
        simulation_time,
    )

# ------------------------------------------------------------
# 5. Conversione dei risultati
# ------------------------------------------------------------

time_history = np.asarray(time_history)
state_history = np.asarray(state_history)
BRT_history = np.asarray(BRT_history)
V0_history = np.asarray(V0_history)
control_history = np.asarray(control_history)
disturbance_history = np.asarray(disturbance_history)
gradient_history = np.asarray(gradient_history)

if len(time_history) == 0:
    raise RuntimeError(
        "La simulazione non ha prodotto alcun campione."
    )

# ------------------------------------------------------------
# 6. Stampa dei risultati
# ------------------------------------------------------------

collision_reached = np.any(V0_history <= 0.0)
full_horizon_completed = (
    termination_reason == "maximum simulation time reached"
    and np.isclose(time_history[-1], simulation_time)
)

print("\n" + "=" * 60)
print("RISULTATI DELLA SIMULAZIONE")
print("=" * 60)

print(f"Motivo terminazione: {termination_reason}")
print(f"Tempo simulato:      {time_history[-1]:.3f} s")
print(f"Numero di campioni:  {len(time_history)}")

print("\nValue function:")
print(f"  BRT iniziale = {BRT_history[0]: .6f}")
print(f"  BRT finale   = {BRT_history[-1]: .6f}")
print(f"  BRT minimo   = {np.min(BRT_history): .6f}")

print("\nCondizione terminale:")
print(f"  V0 iniziale  = {V0_history[0]: .6f}")
print(f"  V0 finale    = {V0_history[-1]: .6f}")
print(f"  V0 minimo    = {np.min(V0_history): .6f}")

print("\nStato finale:")
for name, value in zip(
    coordinate_names,
    state_history[-1],
):
    print(f"  {name:10s} = {value: .6f}")

print("\nUltimo controllo Ego:")
print(
    f"  steering_rate_E = "
    f"{control_history[-1, 0]: .6f} rad/s"
)
print(
    f"  acceleration_E  = "
    f"{control_history[-1, 1]: .6f} m/s²"
)

print("\nUltimo disturbo Human:")
print(
    f"  yaw_rate_H      = "
    f"{disturbance_history[-1, 0]: .6f} rad/s"
)
print(
    f"  acceleration_H  = "
    f"{disturbance_history[-1, 1]: .6f} m/s²"
)

print("\n" + "=" * 60)

if full_horizon_completed and not collision_reached:
    print(
        "TEST SUPERATO: il punto aveva BRT positivo e Ego ha "
        f"evitato la collisione per tutti i {simulation_time:.2f} s."
    )
elif collision_reached:
    print(
        "TEST NON SUPERATO: la traiettoria ha raggiunto "
        f"il terminal set dopo {time_history[-1]:.3f} s."
    )
else:
    print(
        "TEST INCONCLUSIVO: la traiettoria è uscita dalla "
        "griglia prima di completare l'orizzonte."
    )

print("=" * 60)

STATO INIZIALE CASUALE ESTERNO AL BRT
Candidati disponibili: 525,572
Indici selezionati: (np.int64(15), np.int64(8), np.int64(5), np.int64(5), np.int64(4), np.int64(4))
  x_rel      =  7.000000
  y_rel      =  2.000000
  theta_rel  = -0.261799
  v_H        =  6.000000
  delta_E    = -0.052360
  v_E        =  5.000000

BRT(x0) = 2.164669
V0(x0)  = 2.449793

RISULTATI DELLA SIMULAZIONE
Motivo terminazione: state left the interpolation grid
Tempo simulato:      0.570 s
Numero di campioni:  58

Value function:
  BRT iniziale =  2.164669
  BRT finale   =  2.714048
  BRT minimo   =  2.164669

Condizione terminale:
  V0 iniziale  =  2.449793
  V0 finale    =  2.714772
  V0 minimo    =  2.449793

Stato finale:
  x_rel      =  7.316376
  y_rel      =  1.468713
  theta_rel  = -0.291276
  v_H        =  2.010000
  delta_E    = -0.048010
  v_E        =  1.010000

Ultimo controllo Ego:
  steering_rate_E =  0.087000 rad/s
  acceleration_E  = -7.000000 m/s²

Ultimo disturbo Human:
  yaw_rate_H      = 